# Silver · pesquisa

**Origem:** `vcredit_bronze.pesquisa` → **Destino:** `vcredit_silver.pesquisa`

**Grão:** uma linha por resposta de pesquisa. **Depende de:** bronze

Respostas da pesquisa de satisfação, com a nota (1 a 5) dada pelo cliente a cada chamado.

| Regra | Tratamento |
|---|---|
| `id_pesquisa` e `id_chamado` | Convertidos para número |
| `nota_atendimento` | Convertida para número; "NULL", valores menores ou iguais a zero e maiores que 5 viram nulo |
| Mais de uma resposta para o mesmo chamado | Preservadas e contadas em `qtd_chamados_pesquisa` |

**Decisões:** alguns chamados têm duas respostas, sempre com a mesma nota e com `id_pesquisa` diferente. Elas não são removidas aqui, para manter a evidência da inconsistência; a gold usa uma nota por chamado, para que esses chamados não pesem em dobro na média. A nota 0 é tratada como inválida, porque a escala da pesquisa vai de 1 a 5.

Pipeline idempotente (lê da bronze e sobrescreve).

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "rocket_lab"

bronze_schema_name = "vcredit_bronze"
bronze_schema = f"{catalog}.{bronze_schema_name}"

silver_schema_name = "vcredit_silver"
silver_schema = f"{catalog}.{silver_schema_name}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

In [0]:
# Importação da tabela pesquisa da camada bronze
bronze_pesquisa = spark.table(f"{bronze_schema}.pesquisa")

# Particionamento por id_chamado
w = Window.partitionBy("id_chamado")

# Tratamento e padronização dos dados da tabela pesquisa
silver_pesquisa = (
    bronze_pesquisa
    .withColumn("id_pesquisa", F.expr("try_cast(trim(id_pesquisa) as bigint)"))
    .withColumn("id_chamado", F.expr("try_cast(trim(id_chamado) as bigint)"))
    .withColumn("qtd_chamados_pesquisa", F.count("*").over(w))
    .withColumn("nota_atendimento", F.expr("try_cast(nullif(upper(trim(nota_atendimento)), 'NULL') as int)"))
    .withColumn("nota_atendimento", F.when((F.col("nota_atendimento") <= 0) | (F.col("nota_atendimento") > 5), None).otherwise(F.col("nota_atendimento")))
)

 # Idempotente
(
    silver_pesquisa.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.pesquisa")
)

display(silver_pesquisa.orderBy("id_pesquisa").limit(10))